In [3]:
from pathlib import Path
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 0. 기본 경로 설정
# =========================================================

OUTPUT_DIR = Path("outputs")
RESULT_DIR = Path("final_results")

subdirs = {
    "01": RESULT_DIR / "01_region_policy_supply_score",
    "02": RESULT_DIR / "02_region_policy_gap_score",
    "03": RESULT_DIR / "03_policy_category_distribution",
    "04": RESULT_DIR / "04_region_tfidf_keywords",
    "05": RESULT_DIR / "05_kmeans_clustering",
}

RESULT_DIR.mkdir(exist_ok=True)
for d in subdirs.values():
    d.mkdir(parents=True, exist_ok=True)

print("결과 정리 폴더 생성 완료:", RESULT_DIR.resolve())


# =========================================================
# 1. 한글 폰트 설정
# =========================================================

plt.rcParams["axes.unicode_minus"] = False

try:
    plt.rcParams["font.family"] = "Malgun Gothic"
except:
    pass


# =========================================================
# 2. 보조 함수
# =========================================================

def read_csv_auto(path):
    """utf-8-sig, utf-8, cp949 순서로 CSV 읽기."""
    for enc in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            continue
    raise ValueError(f"CSV 파일을 읽을 수 없습니다: {path}")


def find_col(df, candidates, required=True):
    """후보 컬럼명 중 실제 존재하는 컬럼 찾기."""
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"필요 컬럼을 찾지 못했습니다. 후보={candidates}, 실제컬럼={list(df.columns)}")
    return None


def save_interpretation(path, text):
    """해석 텍스트 저장."""
    Path(path).write_text(text, encoding="utf-8-sig")


def make_barh(df, label_col, value_col, title, xlabel, out_path, top_n=10, ascending=True):
    """가로 막대그래프 저장."""
    plot_df = df[[label_col, value_col]].dropna().copy()
    plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[value_col])

    if ascending:
        plot_df = plot_df.sort_values(value_col, ascending=True).tail(top_n)
    else:
        plot_df = plot_df.sort_values(value_col, ascending=False).head(top_n).sort_values(value_col, ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(plot_df[label_col].astype(str), plot_df[value_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# =========================================================
# 1번. 지역별 정책공급종합점수 순위 정리
# =========================================================

score_path = OUTPUT_DIR / "policy_region_score.csv"
score_df = read_csv_auto(score_path)

region_col = find_col(score_df, ["지역", "region"])
supply_col = find_col(score_df, [
    "정책공급종합점수",
    "정책공급점수",
    "종합점수",
    "policy_supply_score",
    "supply_score"
], required=False)

if supply_col is None:
    print("[주의] 정책공급종합점수 컬럼을 찾지 못했습니다.")
    print("policy_region_score.csv 컬럼:", list(score_df.columns))
else:
    supply_rank = score_df.copy()
    supply_rank[supply_col] = pd.to_numeric(supply_rank[supply_col], errors="coerce")
    supply_rank = supply_rank.sort_values(supply_col, ascending=False)

    supply_out_csv = subdirs["01"] / "region_policy_supply_score_rank.csv"
    supply_rank.to_csv(supply_out_csv, index=False, encoding="utf-8-sig")

    make_barh(
        supply_rank,
        label_col=region_col,
        value_col=supply_col,
        title="지역별 정책공급종합점수 순위",
        xlabel="정책공급종합점수",
        out_path=subdirs["01"] / "region_policy_supply_score_rank.png",
        top_n=len(supply_rank),
        ascending=True
    )

    top_region = supply_rank.iloc[0][region_col]
    top_score = supply_rank.iloc[0][supply_col]
    low_region = supply_rank.iloc[-1][region_col]
    low_score = supply_rank.iloc[-1][supply_col]

    save_interpretation(
        subdirs["01"] / "interpretation.txt",
        f"""1. 지역별 정책공급종합점수 순위 해석

이 결과는 지역별 청년정책 공급 수준을 종합점수로 비교한 것이다.
정책공급종합점수는 정책 수, 관련정책 비율, 키워드 대응도, 정책분류 다양성, 신청 가능성, 정보 접근성 등을 종합해 계산한 값이다.

가장 높은 지역:
- {top_region}: {top_score:.2f}점

가장 낮은 지역:
- {low_region}: {low_score:.2f}점

해석:
정책공급종합점수가 높은 지역은 온통청년 정책 데이터 기준으로 정책 수와 정책 정보 제공 수준이 상대적으로 풍부한 지역으로 볼 수 있다.
반대로 점수가 낮은 지역은 정책 수, 정책 분야 다양성, 신청 정보 제공 측면에서 상대적으로 보완이 필요할 가능성이 있다.

주의:
이 점수는 온통청년 정책 텍스트와 정책 메타데이터를 기준으로 계산한 상대적 지표이며, 실제 지역 청년 체감도나 정책 효과를 직접 측정한 값은 아니다.
"""
    )

    print("1번 정리 완료:", supply_out_csv)


# =========================================================
# 2번. 지역별 정책대응부족도 순위 정리
# =========================================================

gap_col = find_col(score_df, [
    "정책대응부족도",
    "정책부족도",
    "대응부족도",
    "policy_gap_score",
    "gap_score"
], required=False)

if gap_col is None:
    if supply_col is not None:
        score_df["정책대응부족도_계산"] = 100 - pd.to_numeric(score_df[supply_col], errors="coerce")
        gap_col = "정책대응부족도_계산"
    else:
        print("[주의] 정책대응부족도 컬럼도, 정책공급종합점수 컬럼도 찾지 못했습니다.")

if gap_col is not None:
    gap_rank = score_df.copy()
    gap_rank[gap_col] = pd.to_numeric(gap_rank[gap_col], errors="coerce")
    gap_rank = gap_rank.sort_values(gap_col, ascending=False)

    gap_out_csv = subdirs["02"] / "region_policy_gap_rank.csv"
    gap_rank.to_csv(gap_out_csv, index=False, encoding="utf-8-sig")

    make_barh(
        gap_rank,
        label_col=region_col,
        value_col=gap_col,
        title="지역별 정책대응부족도 순위",
        xlabel="정책대응부족도",
        out_path=subdirs["02"] / "region_policy_gap_rank.png",
        top_n=len(gap_rank),
        ascending=True
    )

    top_gap_region = gap_rank.iloc[0][region_col]
    top_gap_score = gap_rank.iloc[0][gap_col]
    low_gap_region = gap_rank.iloc[-1][region_col]
    low_gap_score = gap_rank.iloc[-1][gap_col]

    interp_path = OUTPUT_DIR / "policy_region_interpretation.csv"
    if interp_path.exists():
        interp_df = read_csv_auto(interp_path)
        interp_df.to_csv(subdirs["02"] / "policy_region_interpretation.csv", index=False, encoding="utf-8-sig")

    save_interpretation(
        subdirs["02"] / "interpretation.txt",
        f"""2. 지역별 정책대응부족도 순위 해석

이 결과는 지역별 정책 대응이 상대적으로 부족한 정도를 나타낸다.
정책대응부족도가 높을수록 온통청년 정책 텍스트 기준으로 일자리 문제 유형에 대응하는 정책 공급이 상대적으로 부족한 지역으로 해석할 수 있다.

정책대응부족도가 가장 높은 지역:
- {top_gap_region}: {top_gap_score:.2f}점

정책대응부족도가 가장 낮은 지역:
- {low_gap_region}: {low_gap_score:.2f}점

해석:
정책대응부족도가 높은 지역은 고용기회, 임금소득, 직무성장, 워라밸환경, 주거안정, 창업생태계, 참여관계 등의 문제 유형 중 일부에 대응하는 정책 키워드가 상대적으로 부족할 가능성이 있다.
따라서 해당 지역은 세부 문제유형별 점수를 함께 확인하여 어떤 정책 분야가 보완되어야 하는지 살펴볼 필요가 있다.

주의:
이 부족도는 실제 정책 효과가 아니라 정책 텍스트에 나타난 키워드 대응성을 기반으로 한 분석 결과이다.
"""
    )

    print("2번 정리 완료:", gap_out_csv)


# =========================================================
# 3번. 정책 분류별 분포 정리
# =========================================================

cat_summary_path = OUTPUT_DIR / "policy_category_summary.csv"
cat_df = read_csv_auto(cat_summary_path)

cat_col = find_col(cat_df, ["정책분류", "대표분류", "분류", "정책분야", "category"], required=False)
count_col = find_col(cat_df, ["정책수", "건수", "count", "cnt"], required=False)

cat_dist = None

if cat_col is None or count_col is None:
    preprocessed = read_csv_auto(OUTPUT_DIR / "policy_preprocessed.csv")
    cat_col2 = find_col(preprocessed, ["정책분류", "대표분류", "분류", "정책분야", "category"], required=False)

    if cat_col2 is None:
        print("[주의] 정책 분류 컬럼을 찾지 못했습니다.")
        print("policy_preprocessed.csv 컬럼:", list(preprocessed.columns))
    else:
        cat_dist = (
            preprocessed[cat_col2]
            .fillna("기타")
            .value_counts()
            .reset_index()
        )
        cat_dist.columns = ["정책분류", "정책수"]
else:
    cat_dist = cat_df[[cat_col, count_col]].copy()
    cat_dist.columns = ["정책분류", "정책수"]

if cat_dist is not None:
    cat_dist["정책수"] = pd.to_numeric(cat_dist["정책수"], errors="coerce")
    cat_dist = cat_dist.dropna(subset=["정책수"]).sort_values("정책수", ascending=False)
    cat_dist["비율(%)"] = (cat_dist["정책수"] / cat_dist["정책수"].sum() * 100).round(2)

    cat_out_csv = subdirs["03"] / "policy_category_distribution.csv"
    cat_dist.to_csv(cat_out_csv, index=False, encoding="utf-8-sig")

    make_barh(
        cat_dist,
        label_col="정책분류",
        value_col="정책수",
        title="정책 분류별 분포",
        xlabel="정책 수",
        out_path=subdirs["03"] / "policy_category_distribution.png",
        top_n=len(cat_dist),
        ascending=True
    )

    pivot_path = OUTPUT_DIR / "policy_region_category_pivot.csv"
    if pivot_path.exists():
        shutil.copy(pivot_path, subdirs["03"] / "policy_region_category_pivot.csv")

    top_cat = cat_dist.iloc[0]["정책분류"]
    top_cat_count = cat_dist.iloc[0]["정책수"]
    top_cat_ratio = cat_dist.iloc[0]["비율(%)"]

    save_interpretation(
        subdirs["03"] / "interpretation.txt",
        f"""3. 정책 분류별 분포 해석

이 결과는 온통청년 정책 데이터가 어떤 정책 분야에 많이 분포되어 있는지를 보여준다.

가장 많은 정책 분야:
- {top_cat}: {int(top_cat_count)}건, 전체의 {top_cat_ratio:.2f}%

해석:
정책 분류별 분포를 보면 온통청년 정책이 특정 분야에 집중되어 있는지, 또는 여러 정책 분야에 고르게 분포되어 있는지 확인할 수 있다.
예를 들어 일자리 또는 직무교육 정책이 많고 주거지원, 창업지원, 참여 프로그램 정책이 적다면 정책 공급이 일부 분야에 편중되어 있다고 해석할 수 있다.

활용:
이 결과는 지역별 정책대응부족도 해석과 함께 사용하면 좋다.
정책 수가 많은 분야와 부족한 분야를 함께 비교하면 정책 공급의 균형성을 설명할 수 있다.
"""
    )

    print("3번 정리 완료:", cat_out_csv)


# =========================================================
# 4번. 지역별 TF-IDF 핵심 키워드 정리
# =========================================================

tfidf_path = OUTPUT_DIR / "region_tfidf_keywords.csv"
tfidf_df = read_csv_auto(tfidf_path)

tfidf_region_col = find_col(tfidf_df, ["지역", "region"])
keyword_col = find_col(tfidf_df, ["키워드", "단어", "term", "keyword"])
tfidf_score_col = find_col(
    tfidf_df,
    ["tfidf", "TF-IDF", "TFIDF", "tfidf_score", "tf_idf", "점수", "score"],
    required=False
)

if tfidf_score_col is None:
    print("[주의] TF-IDF 점수 컬럼을 찾지 못했습니다.")
    print("region_tfidf_keywords.csv 컬럼:", list(tfidf_df.columns))
else:
    tfidf_df[tfidf_score_col] = pd.to_numeric(tfidf_df[tfidf_score_col], errors="coerce")
    tfidf_df = tfidf_df.dropna(subset=[tfidf_score_col])

    region_tfidf_top = (
        tfidf_df.sort_values([tfidf_region_col, tfidf_score_col], ascending=[True, False])
        .groupby(tfidf_region_col)
        .head(10)
        .reset_index(drop=True)
    )

    tfidf_out_csv = subdirs["04"] / "region_tfidf_top_keywords.csv"
    region_tfidf_top.to_csv(tfidf_out_csv, index=False, encoding="utf-8-sig")

    lines = ["4. 지역별 TF-IDF 핵심 키워드\n"]
    for region, part in region_tfidf_top.groupby(tfidf_region_col):
        keywords = ", ".join(part[keyword_col].astype(str).tolist())
        lines.append(f"[{region}] {keywords}")

    save_interpretation(
        subdirs["04"] / "region_tfidf_top_keywords_by_region.txt",
        "\n".join(lines)
    )

    top1 = (
        tfidf_df.sort_values([tfidf_region_col, tfidf_score_col], ascending=[True, False])
        .groupby(tfidf_region_col)
        .head(1)
        .copy()
    )
    top1["지역_키워드"] = top1[tfidf_region_col].astype(str) + " - " + top1[keyword_col].astype(str)

    make_barh(
        top1,
        label_col="지역_키워드",
        value_col=tfidf_score_col,
        title="지역별 TF-IDF 1위 핵심 키워드",
        xlabel="TF-IDF 점수",
        out_path=subdirs["04"] / "region_tfidf_top_keywords.png",
        top_n=len(top1),
        ascending=True
    )

    save_interpretation(
        subdirs["04"] / "interpretation.txt",
        f"""4. 지역별 TF-IDF 핵심 키워드 해석

TF-IDF는 단순히 많이 등장한 단어가 아니라, 특정 지역에서 상대적으로 중요하게 나타나는 단어를 찾는 방법이다.
따라서 지역별 TF-IDF 핵심 키워드는 각 지역 청년정책의 특징을 보여준다.

해석 방법:
- 특정 지역에서 '취업', '채용', '일자리'가 높으면 고용기회 관련 정책 특성이 강하다고 볼 수 있다.
- '교육', '훈련', '자격증', '역량'이 높으면 직무교육 또는 성장지원 정책 특성이 강하다고 볼 수 있다.
- '주거', '월세', '임대', '전세'가 높으면 주거지원 정책 특성이 강하다고 볼 수 있다.
- '창업', '사업화', '스타트업'이 높으면 창업지원 정책 특성이 강하다고 볼 수 있다.

주의:
TF-IDF 점수는 지역 간 상대적 중요도를 보여주는 값이므로, 실제 정책 효과나 예산 규모를 직접 의미하지는 않는다.
"""
    )

    print("4번 정리 완료:", tfidf_out_csv)


# =========================================================
# 5번. K-means 클러스터링 결과 정리
# =========================================================

cluster_size_path = OUTPUT_DIR / "kmeans_cluster_size.csv"
cluster_keywords_path = OUTPUT_DIR / "kmeans_cluster_keywords.csv"
cluster_examples_path = OUTPUT_DIR / "kmeans_cluster_examples.csv"
cluster_result_path = OUTPUT_DIR / "policy_textmining_kmeans_result.csv"

# 원본 결과 파일 복사
for src in [cluster_size_path, cluster_keywords_path, cluster_examples_path, cluster_result_path]:
    if src.exists():
        shutil.copy(src, subdirs["05"] / src.name)

if cluster_size_path.exists():
    cluster_size = read_csv_auto(cluster_size_path)

    print("kmeans_cluster_size.csv 컬럼:", list(cluster_size.columns))

    cluster_col = find_col(
        cluster_size,
        [
            "cluster",
            "클러스터",
            "군집",
            "Cluster",
            "KMeans클러스터",
            "KMeans군집",
            "kmeans_cluster",
            "KMeansCluster"
        ]
    )

    cluster_count_col = find_col(
        cluster_size,
        [
            "정책수",
            "건수",
            "count",
            "cnt",
            "문서수",
            "데이터수"
        ],
        required=False
    )

    if cluster_count_col is None:
        cluster_count_col = [c for c in cluster_size.columns if c != cluster_col][0]

    cluster_size[cluster_count_col] = pd.to_numeric(cluster_size[cluster_count_col], errors="coerce")
    cluster_size = cluster_size.dropna(subset=[cluster_count_col])
    cluster_size = cluster_size.sort_values(cluster_col)

    cluster_size.to_csv(
        subdirs["05"] / "cluster_size_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    plt.figure(figsize=(8, 5))
    plt.bar(cluster_size[cluster_col].astype(str), cluster_size[cluster_count_col])
    plt.title("K-means 클러스터별 정책 수")
    plt.xlabel("클러스터")
    plt.ylabel("정책 수")
    plt.tight_layout()
    plt.savefig(subdirs["05"] / "cluster_distribution.png", dpi=200)
    plt.close()

    cluster_summary_text = "5. K-means 클러스터링 결과 해석\n\n"
    cluster_summary_text += "K-means 클러스터링은 정책 텍스트를 TF-IDF 벡터로 변환한 뒤, 텍스트가 비슷한 정책끼리 자동으로 묶은 결과이다.\n\n"
    cluster_summary_text += "클러스터별 정책 수:\n"

    for _, row in cluster_size.iterrows():
        cluster_summary_text += f"- 클러스터 {row[cluster_col]}: {int(row[cluster_count_col])}건\n"

    if cluster_keywords_path.exists():
        cluster_keywords = read_csv_auto(cluster_keywords_path)
        cluster_keywords.to_csv(
            subdirs["05"] / "cluster_top_keywords.csv",
            index=False,
            encoding="utf-8-sig"
        )
        cluster_summary_text += "\n클러스터별 대표 키워드는 cluster_top_keywords.csv 파일을 확인한다.\n"

    if cluster_examples_path.exists():
        cluster_examples = read_csv_auto(cluster_examples_path)
        cluster_examples.to_csv(
            subdirs["05"] / "cluster_representative_policies.csv",
            index=False,
            encoding="utf-8-sig"
        )
        cluster_summary_text += "클러스터별 대표 정책 사례는 cluster_representative_policies.csv 파일을 확인한다.\n"

    cluster_summary_text += """
해석 방법:
각 클러스터의 대표 키워드와 대표 정책명을 확인한 뒤, 사람이 의미를 붙이면 된다.
예를 들어 대표 키워드가 '취업, 채용, 구직, 일자리' 중심이면 '취업·고용지원형'으로 해석할 수 있다.
대표 키워드가 '주거, 월세, 임대, 전세' 중심이면 '주거지원형'으로 해석할 수 있다.
대표 키워드가 '교육, 훈련, 자격증, 역량' 중심이면 '직무교육·역량개발형'으로 해석할 수 있다.

주의:
K-means 클러스터 번호 자체에는 의미가 없다.
예를 들어 클러스터 0이 항상 일자리 정책이라는 뜻은 아니며, 반드시 대표 키워드와 대표 정책명을 보고 해석해야 한다.
"""

    save_interpretation(subdirs["05"] / "interpretation.txt", cluster_summary_text)

    print("5번 정리 완료:", subdirs["05"])
else:
    print("[주의] kmeans_cluster_size.csv 파일을 찾지 못했습니다.")

# =========================================================
# 전체 README 작성
# =========================================================

readme_text = """최종 분석 결과 정리 폴더 안내

이 폴더는 온통청년 청년정책 텍스트마이닝 소프로젝트의 핵심 결과를 5개 항목으로 정리한 것이다.

1. 01_region_policy_supply_score
- 지역별 정책공급종합점수 순위를 정리하였다.
- 정책 공급 수준이 높은 지역과 낮은 지역을 비교할 수 있다.

2. 02_region_policy_gap_score
- 지역별 정책대응부족도 순위를 정리하였다.
- 정책 키워드 대응이 상대적으로 부족한 지역을 확인할 수 있다.

3. 03_policy_category_distribution
- 정책 분류별 분포를 정리하였다.
- 일자리, 직무교육, 주거지원, 창업지원, 복지, 참여 프로그램 등 정책 분야별 비중을 확인할 수 있다.

4. 04_region_tfidf_keywords
- 지역별 TF-IDF 핵심 키워드를 정리하였다.
- 각 지역 정책 텍스트에서 상대적으로 중요한 키워드를 확인할 수 있다.

5. 05_kmeans_clustering
- TF-IDF 기반 K-means 클러스터링 결과를 정리하였다.
- 텍스트가 유사한 정책끼리 어떤 군집으로 묶이는지 확인할 수 있다.

보고서/PPT에 우선적으로 사용할 자료:
- 01_region_policy_supply_score/region_policy_supply_score_rank.png
- 02_region_policy_gap_score/region_policy_gap_rank.png
- 03_policy_category_distribution/policy_category_distribution.png
- 04_region_tfidf_keywords/region_tfidf_top_keywords.png
- 05_kmeans_clustering/cluster_distribution.png

주의:
본 분석은 온통청년 정책 텍스트와 정책 메타데이터를 기반으로 한 분석이다.
따라서 실제 정책 효과, 예산 규모, 청년 체감 만족도를 직접 측정한 결과는 아니다.
"""

save_interpretation(RESULT_DIR / "README_result_summary.txt", readme_text)

print("\n전체 결과 정리 완료")
print("생성 위치:", RESULT_DIR.resolve())

결과 정리 폴더 생성 완료: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\final_results
1번 정리 완료: final_results\01_region_policy_supply_score\region_policy_supply_score_rank.csv
2번 정리 완료: final_results\02_region_policy_gap_score\region_policy_gap_rank.csv
3번 정리 완료: final_results\03_policy_category_distribution\policy_category_distribution.csv
4번 정리 완료: final_results\04_region_tfidf_keywords\region_tfidf_top_keywords.csv
kmeans_cluster_size.csv 컬럼: ['KMeans클러스터', '정책수']
5번 정리 완료: final_results\05_kmeans_clustering

전체 결과 정리 완료
생성 위치: C:\Users\yong\Desktop\TM\tm2\textmining_solo_policy_project\textmining_solo_policy_project\final_results
